# Run uploaded CUDA SumCheck directory

Use this notebook in Colab/H100 after uploading `cuda_sumcheck_current.zip`. It unzips, builds the CUDA extension, runs a smoke test, and shows where to copy files into an assignment harness.


In [ ]:
from google.colab import files
uploaded = files.upload()
print(uploaded.keys())


In [ ]:
import zipfile, pathlib, os, shutil
zip_name = next(name for name in uploaded if name.endswith('.zip'))
root = pathlib.Path('/content/cuda_sumcheck_current')
if root.exists():
    shutil.rmtree(root)
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall('/content')
# Handle either top-level folder or flat zip.
if not root.exists():
    candidates = [p for p in pathlib.Path('/content').iterdir() if p.is_dir() and (p/'student.py').exists()]
    root = candidates[0]
print('ROOT =', root)
os.chdir(root)


In [ ]:
!python --version
!nvidia-smi
!python - <<'PY'
import torch
print('torch', torch.__version__, 'cuda available', torch.cuda.is_available())
print('cuda', torch.version.cuda)
PY


In [ ]:
!python setup.py build_ext --inplace


In [ ]:
!python scripts/run_smoke.py --cuda


## Optional: install into an uploaded harness

Upload or mount your assignment directory, then set `HARNESS_ROOT` and run the cell below.


In [ ]:
HARNESS_ROOT = '/content/sumcheck_repo/assignment2'  # change this
!python scripts/install_in_harness.py {HARNESS_ROOT}
%cd {HARNESS_ROOT}
!python setup.py build_ext --inplace
!python - <<'PY'
import student
print(student.native_status())
PY
